In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})

In [ ]:
def load_run(log_path):
    """Parse a single task .out file → (config dict, list of eval dicts)."""
    config, evals = None, []
    for line in Path(log_path).read_text().splitlines():
        line = line.strip()
        if not line:
            continue
        try:
            obj = json.loads(line)
        except json.JSONDecodeError:
            continue
        if obj.get("event") == "config":
            config = obj
        elif obj.get("event") == "eval":
            evals.append(obj)
    return config, evals


def load_sweep(group):
    """Load all runs for a sweep group. Returns list of (config, evals) tuples."""
    log_dir = Path(f"../logs/{group}/run_info/logs")
    runs = []
    for f in sorted(log_dir.glob("*.out")):
        config, evals = load_run(f)
        if config is not None and evals:
            runs.append((config, evals))
    return runs

## Sweep 1 — LR sensitivity (AdamW, η ∈ {3e-5, 1e-4, 3e-4, 1e-3})

In [ ]:
lr_runs = load_sweep("sweep_lr2")
print(f"Loaded {len(lr_runs)} runs")
for cfg, evs in lr_runs:
    lr = cfg["command"].split("--lr")[1].split()[0]
    print(f"  η={lr}  steps={[e['step'] for e in evs]}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

colors = plt.cm.viridis([0.1, 0.35, 0.65, 0.9])
lr_runs_sorted = sorted(lr_runs, key=lambda x: float(x[0]["command"].split("--lr")[1].split()[0]))

for (cfg, evs), color in zip(lr_runs_sorted, colors):
    lr = cfg["command"].split("--lr")[1].split()[0]
    steps = [e["step"] for e in evs]
    eval_losses = [e["eval_loss"] for e in evs]
    ax.plot(steps, eval_losses, marker="o", color=color, label=f"η={lr}")

ax.set_xlabel("Step")
ax.set_ylabel("Eval loss")
ax.legend()
ax.xaxis.set_major_locator(ticker.MultipleLocator(100))
ax.grid(True, alpha=0.3)
ax.set_title("LR sweep — AdamW, OLMo-2-0425-1B, Magicoder, r=16")
fig.tight_layout()
plt.show()

## Sweep 2 — LoRA+ multiplier (AdamW, η=1e-4, m ∈ {1, 4, 16, 32})

In [ ]:
lp_runs = load_sweep("loraplus_sweep")
print(f"Loaded {len(lp_runs)} runs")
for cfg, evs in lp_runs:
    m = cfg["command"].split("--lora_plus_multiplier")[1].split()[0]
    print(f"  m={m}  steps={[e['step'] for e in evs]}")

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))

colors = plt.cm.plasma([0.1, 0.37, 0.63, 0.9])
lp_runs_sorted = sorted(
    lp_runs,
    key=lambda x: float(x[0]["command"].split("--lora_plus_multiplier")[1].split()[0]),
)

for (cfg, evs), color in zip(lp_runs_sorted, colors):
    m = cfg["command"].split("--lora_plus_multiplier")[1].split()[0]
    steps = [e["step"] for e in evs]
    eval_losses = [e["eval_loss"] for e in evs]
    ax.plot(steps, eval_losses, marker="o", color=color, label=f"m={m}")

ax.set_xlabel("Step")
ax.set_ylabel("Eval loss")
ax.legend()
ax.xaxis.set_major_locator(ticker.MultipleLocator(100))
ax.grid(True, alpha=0.3)
ax.set_title("LoRA+ sweep — AdamW η=1e-4, OLMo-2-0425-1B, Magicoder, r=16")
fig.tight_layout()
plt.show()